In [16]:
# Essential library imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib.lines import Line2D
# Suppress technical warnings and file paths in output
import warnings
warnings.filterwarnings('ignore')

# Global environment configuration
pd.set_option("display.max_columns", None)

In [17]:
df_ess = pd.read_csv("original-csv/ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.csv", sep=';', low_memory=False) # Load dataset

display(df_ess.head(2)) # ESS dataset exploration
print("=" * 100)

display(df_ess.tail(2))
print("=" * 100)

display(df_ess.sample(2))
print(f"ESS dataframe has {df_ess.shape[0]} rows and {df_ess.shape[1]} columns")

,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,prob,stratum,psu,actrolga,trstun,imbgeco,imueclt,imwbcnt,volunfp,donprty,brncntr,cntbrthd,ctzcntr,rlgdgr,sclact,ccnthum,ccrdprs,wrclmch,gndr,eisced,hincfel,maritalb,tporgwk,ipeqopt,ipeqopta,yrbrn
0,ESS9e03_3,9,3.3,26.01.2026,27,AT,0.5811743,0.21811137,0.30209145,0.06588958,0.001176,59,1688,2,5,5,6,0,NaN,NaN,1,6666,1,0,3,NaN,NaN,NaN,1,3,3,4.0,4,2.0,NaN,1975
1,ESS9e03_3,9,3.3,26.01.2026,137,AT,10.627.724,0.41347328,0.30209145,0.12490674,0.000643,79,88,2,2,5,4,5,NaN,NaN,1,6666,1,5,3,NaN,NaN,NaN,1,3,2,1.0,4,3.0,NaN,1951


,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,prob,stratum,psu,actrolga,trstun,imbgeco,imueclt,imwbcnt,volunfp,donprty,brncntr,cntbrthd,ctzcntr,rlgdgr,sclact,ccnthum,ccrdprs,wrclmch,gndr,eisced,hincfel,maritalb,tporgwk,ipeqopt,ipeqopta,yrbrn
159318,ESS11e04_1,11,4.1,12.01.2026,273838,UA,0.5974121,0.5974121,13.107.032,0.78303,0.000369,1492,25303,1,0,88,6,3,2.0,2.0,1,6666,1,5,1,4.0,1.0,3.0,1,1,4,5.0,6,NaN,4.0,1947
159319,ESS11e04_1,11,4.1,12.01.2026,273840,UA,0.8442026,0.8442026,13.107.032,1.106.499,0.000261,1527,25227,1,8,5,5,5,2.0,2.0,1,6666,1,10,3,4.0,5.0,3.0,2,7,2,1.0,3,NaN,1.0,1965


,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,prob,stratum,psu,actrolga,trstun,imbgeco,imueclt,imwbcnt,volunfp,donprty,brncntr,cntbrthd,ctzcntr,rlgdgr,sclact,ccnthum,ccrdprs,wrclmch,gndr,eisced,hincfel,maritalb,tporgwk,ipeqopt,ipeqopta,yrbrn
24793,ESS9e03_3,9,3.3,26.01.2026,5492,HR,0.99777675,0.69978446,0.19398342,0.13574658,0.001040,1591,12771,2,8,3,6,4,NaN,NaN,1,6666,1,1,3,NaN,NaN,NaN,2,4,1,6.0,66,1.0,NaN,1996
21569,ESS9e03_3,9,3.3,26.01.2026,38483,FR,1.139.468,10.964.046,27.266.867,29.895.518,0.000076,1391,11956,2,5,8,8,8,NaN,NaN,1,6666,1,5,3,NaN,NaN,NaN,2,2,2,1.0,4,3.0,NaN,1940


ESS dataframe has 159320 rows and 36 columns


In [18]:
# Detailed information 
print("--- Data Summary Table ---")
survey_data_summary = pd.DataFrame({
    'Data Type': df_ess.dtypes,
    'Non-Null Count': df_ess.count(),
    'Null Count': df_ess.isnull().sum(),
    'Null Percentage (%)': (df_ess.isnull().sum() / len(df_ess)) * 100
}).round(2)
display(survey_data_summary)

--- Data Summary Table ---


,Data Type,Non-Null Count,Null Count,Null Percentage (%)
name,str,159320,0,0.00
essround,int64,159320,0,0.00
edition,float64,159320,0,0.00
proddate,str,159320,0,0.00
idno,int64,159320,0,0.00
cntry,str,159320,0,0.00
dweight,str,159320,0,0.00
pspwght,str,159320,0,0.00
pweight,str,159320,0,0.00
anweight,str,159320,0,0.00


In [19]:
# 2. DEFINICIÓN DE CATEGORÍAS (Basado en data_builder_codes.docx)
# Nota: Los códigos marcados como "Missing Value" (7, 8, 9, 77, 88, 99, etc.) 
# NO se incluyen en el diccionario para que .map() los convierta automáticamente en NaN.

mapeos = {
    'actrolga': {1: 'Nada capaz', 2: 'Poco capaz', 3: 'Bastante capaz', 4: 'Muy capaz', 5: 'Completamente capaz'},
    'trstun': {i: str(i) for i in range(11)}, # 0: No trust at all, 10: Complete trust
    'imbgeco': {i: str(i) for i in range(11)}, # 0: Bad, 10: Good for the economy
    'imueclt': {i: str(i) for i in range(11)}, # 0: Undermined, 10: Enriched
    'imwbcnt': {i: str(i) for i in range(11)}, # 0: Worse, 10: Better
    'volunfp': {1: 'Sí', 2: 'No'},
    'donprty': {1: 'Sí', 2: 'No'},
    'brncntr': {1: 'Sí', 2: 'No'},
    'ctzcntr': {1: 'Sí', 2: 'No'},
    'rlgdgr': {i: str(i) for i in range(11)}, # 0: Not religious, 10: Very religious
    'sclact': {1: 'Mucho menos', 2: 'Menos', 3: 'Igual', 4: 'Más', 5: 'Mucho más'},
    'ccrdprs': {i: str(i) for i in range(11)}, # 0: Nada, 10: Mucho
    'ccnthum': {1: 'Procesos naturales', 2: 'Principalmente naturales', 3: 'Ambos por igual', 
                4: 'Principalmente humanos', 5: 'Enteramente humanos', 55: 'No cree que ocurra'},
    'wrclmch': {1: 'Nada preocupado', 2: 'No muy preocupado', 3: 'Algo preocupado', 
                4: 'Muy preocupado', 5: 'Extremadamente preocupado'},
    'gndr': {1: 'Hombre', 2: 'Mujer'},
    'eisced': {
        0: 'No armonizable', 1: 'Primaria o menos', 2: 'Secundaria baja', 3: 'Secundaria alta (baja)',
        4: 'Secundaria alta (alta)', 5: 'Vocacional avanzado', 6: 'Grado/BA', 7: 'Máster/Doctorado', 55: 'Otro'
    },
    'hincfel': {1: 'Cómodamente', 2: 'Saliendo adelante', 3: 'Difícil', 4: 'Muy difícil'},
    'maritalb': {1: 'Casado', 2: 'Unión civil', 3: 'Separado', 4: 'Divorciado', 5: 'Viudo', 6: 'Nunca casado'},
    'tporgwk': {1: 'Gobierno central/local', 2: 'Sector público (Edu/Salud)', 3: 'Empresa estatal', 
                4: 'Empresa privada', 5: 'Autónomo', 6: 'Otro'},
    'ipeqopt': {1: 'Muy parecido a mí', 2: 'Parecido', 3: 'Algo parecido', 4: 'Un poco parecido', 5: 'No como yo', 6: 'Nada como yo'},
    'ipeqopta': {1: 'Muy parecido a mí', 2: 'Parecido', 3: 'Algo parecido', 4: 'Un poco parecido', 5: 'No como yo', 6: 'Nada como yo'}
}

In [20]:
# 3. PROCESAMIENTO Y LIMPIEZA INTEGRAL DE NULOS
df_cat = df_ess.copy()

# Listado completo de códigos de "Missing Values" de la ESS según el ancho de la variable
codigos_nulos = [
    6, 7, 8, 9,          # 1 dígito (incluye 6: Not applicable)
    66, 77, 88, 99,      # 2 dígitos
    6666, 7777, 8888, 9999 # 4 dígitos (Años y códigos de país)
]

# Paso A: Limpiamos TODAS las variables numéricas que no vamos a mapear (como yrbrn o cntbrthd)
# Esto asegura que los 7777 o 99 no distorsionen las medias
columnas_excluir= ["essround"]
for col in df_cat.columns:
    if col not in mapeos.keys() and col not in columnas_excluir:
        df_cat[col] = df_cat[col].replace(codigos_nulos, np.nan)

# Paso B: Aplicamos los mapeos a las variables categóricas
# (Aquí los 7, 8, 9, etc. se vuelven NaN porque no están en los diccionarios de 'mapeos')
for col, diccionario in mapeos.items():
    if col in df_cat.columns:
        df_cat[col] = df_cat[col].map(diccionario)
        df_cat[col] = df_cat[col].astype('category')

In [21]:
df_cat['essround'].value_counts(dropna=False) #para comprobar que seguimos teniendo el 9.


essround
10    59685
11    50116
9     49519
Name: count, dtype: int64

In [22]:
# TABLA RESUMEN DE DATOS (Calidad y Tipos)
print("\n--- Data Summary Table ---")
survey_data_summary = pd.DataFrame({
    'Data Type': df_cat.dtypes,
    'Non-Null Count': df_cat.count(),
    'Null Count': df_cat.isnull().sum(),
    'Null Percentage (%)': (df_cat.isnull().sum() / len(df_cat)) * 100
}).round(2)

# Usamos print para asegurar visibilidad en consola, o display si estás en Notebook
try:
    from IPython.display import display
    display(survey_data_summary)
except ImportError:
    print(survey_data_summary)

print("\n--- RESUMEN DE VARIABLES NUMÉRICAS LIMPIAS ---")
print(df_cat.describe().T)

print("\n--- DISTRIBUCIÓN PORCENTUAL DE VARIABLES CATEGÓRICAS ---")
columnas_interes = ['gndr', 'volunfp', 'donprty', 'eisced']
for col in columnas_interes:
    if col in df_cat.columns:
        print(f"\nFrecuencia en {col}:")
        print(df_cat[col].value_counts(normalize=True).round(4) * 100)

print("\n--- VISTA PREVIA DE LOS DATOS ---")
print(df_cat.head())


--- Data Summary Table ---


,Data Type,Non-Null Count,Null Count,Null Percentage (%)
name,str,159320,0,0.00
essround,int64,159320,0,0.00
edition,float64,159320,0,0.00
proddate,str,159320,0,0.00
idno,float64,159317,3,0.00
cntry,str,159320,0,0.00
dweight,str,159320,0,0.00
pspwght,str,159320,0,0.00
pweight,str,159320,0,0.00
anweight,str,159320,0,0.00



--- RESUMEN DE VARIABLES NUMÉRICAS LIMPIAS ---
             count          mean           std          min           25%  \
essround  159320.0     10.003747      0.790801     9.000000      9.000000   
edition   159320.0      3.537794      0.382309     3.200000      3.300000   
idno      159317.0  48611.632726  34972.757881     2.000000  22175.000000   
prob      159320.0      0.001198      0.002357     0.000005      0.000252   
stratum   158954.0   1202.333235    885.888443     1.000000    357.000000   
psu       159238.0  12541.738712   7287.075808     1.000000   6028.000000   
yrbrn     157367.0   1970.063412     18.721904  1928.000000   1955.000000   

                   50%           75%           max  
essround     10.000000     11.000000      11.00000  
edition       3.300000      4.100000       4.10000  
idno      49264.000000  66116.000000  273842.00000  
prob          0.000549      0.001054       0.04545  
stratum    1036.000000   2110.000000    2914.00000  
psu       12284.0

In [23]:
# CREACIÓN DEL DATAFRAME FINAL 
# Seleccionamos solo las variables relevantes para el análisis de conciencia social
columnas_analisis = {
    'idno': 'ID_Encuestado',
    'cntry': 'Pais',
    'essround': 'Ronda',
    'gndr': 'Genero',
    'yrbrn': 'Anio_Nacimiento',
    'eisced': 'Nivel_Educativo',
    'hincfel': 'Situacion_Economica',
    'volunfp': 'Voluntariado_12m',
    'donprty': 'Donacion_Politica_12m',
    'actrolga': 'Capacidad_Rol_Politico',
    'trstun': 'Confianza_ONU',
    'wrclmch': 'Preocupacion_Clima',
    'ccrdprs': 'Responsabilidad_Clima',
    'imbgeco': 'Inmigracion_Economia',
    'imueclt': 'Inmigracion_Cultura',
    'imwbcnt': 'Inmigracion_Pais_Mejor_Peor'
}

# Filtramos para quedarnos solo con lo que existe en el dataset
columnas_finales = [col for col in columnas_analisis.keys() if col in df_cat.columns]

# Creamos el dataframe limpio
df_final = df_cat[columnas_finales].rename(columns=columnas_analisis)

print("\n" + "="*60)
print("--- DATAFRAME FINAL PARA ANÁLISIS DE EQUIPO ---")
print("="*60)
print(f"Variables seleccionadas: {len(df_final.columns)}")
print(f"Registros disponibles: {len(df_final)}")

# Resumen estadístico final presentado como DataFrame
print("\n--- Estadísticas del Dataset de Conciencia Social ---")
df_stats = df_final.describe(include="all").T

# Añadimos columna de porcentaje de nulos
df_stats['%_Nulos'] = (df_final.isnull().sum() / len(df_final) * 100).round(2)

try:
    display(df_stats)
except:
    print(df_stats)

# Vista previa para las compañeras presentada como DataFrame
print("\n--- Muestra del Dataset Final (Head) ---")
try:
    display(df_final.tail(10))
except:
    print(df_final.tail(10))


--- DATAFRAME FINAL PARA ANÁLISIS DE EQUIPO ---
Variables seleccionadas: 16
Registros disponibles: 159320

--- Estadísticas del Dataset de Conciencia Social ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max,%_Nulos
ID_Encuestado,159317.0,NaN,NaN,NaN,48611.632726,34972.757881,2.0,22175.0,49264.0,66116.0,273842.0,0.00
Pais,159320,33,DE,13503,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
Ronda,159320.0,NaN,NaN,NaN,10.003747,0.790801,9.0,9.0,10.0,11.0,11.0,0.00
Genero,158685,2,Mujer,84812,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.40
Anio_Nacimiento,157367.0,NaN,NaN,NaN,1970.063412,18.721904,1928.0,1955.0,1969.0,1985.0,2009.0,1.23
Nivel_Educativo,157921,8,Secundaria alta (alta),36723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.88
Situacion_Economica,156091,4,Saliendo adelante,70796,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.03
Voluntariado_12m,109068,2,No,88620,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.54
Donacion_Politica_12m,108817,2,No,101876,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.70
Capacidad_Rol_Politico,156238,5,Nada capaz,56844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.93



--- Muestra del Dataset Final (Head) ---


,ID_Encuestado,Pais,Ronda,Genero,Anio_Nacimiento,Nivel_Educativo,Situacion_Economica,Voluntariado_12m,Donacion_Politica_12m,Capacidad_Rol_Politico,Confianza_ONU,Preocupacion_Clima,Responsabilidad_Clima,Inmigracion_Economia,Inmigracion_Cultura,Inmigracion_Pais_Mejor_Peor
159310,273613.0,UA,11,Mujer,1997.0,Vocacional avanzado,Muy difícil,Sí,No,Nada capaz,5,Algo preocupado,0,8,10,10
159311,273621.0,UA,11,Hombre,2008.0,Secundaria alta (alta),Saliendo adelante,No,No,Poco capaz,6,Algo preocupado,5,5,8,8
159312,273688.0,UA,11,Hombre,1983.0,Vocacional avanzado,Saliendo adelante,Sí,No,Poco capaz,5,Muy preocupado,7,8,7,6
159313,273722.0,UA,11,Mujer,1983.0,Máster/Doctorado,Saliendo adelante,Sí,No,Poco capaz,5,Algo preocupado,5,5,5,5
159314,273734.0,UA,11,Hombre,1987.0,Vocacional avanzado,Difícil,No,No,Poco capaz,5,No muy preocupado,3,6,8,5
159315,273742.0,UA,11,Mujer,1973.0,Secundaria alta (alta),Cómodamente,No,No,Nada capaz,7,Algo preocupado,7,5,4,7
159316,273743.0,UA,11,Mujer,1959.0,Vocacional avanzado,Saliendo adelante,No,No,Nada capaz,7,Muy preocupado,9,10,NaN,10
159317,273747.0,UA,11,Hombre,1954.0,Grado/BA,Muy difícil,No,No,Nada capaz,3,Extremadamente preocupado,7,7,6,7
159318,273838.0,UA,11,Hombre,1947.0,Primaria o menos,Muy difícil,No,No,Nada capaz,0,Algo preocupado,1,NaN,6,3
159319,273840.0,UA,11,Mujer,1965.0,Máster/Doctorado,Saliendo adelante,No,No,Nada capaz,8,Algo preocupado,5,5,5,5


In [24]:
mascara_2018 = df_final["Ronda"] == 9
df_ess_2018 = df_final[mascara_2018] 
mascara_2020 = df_final["Ronda"] == 10
df_ess_2020 = df_final[mascara_2020] 
mascara_2023 = df_final["Ronda"] == 11
df_ess_2023 = df_final[mascara_2023] 



In [25]:
display(df_ess_2018.head(4))
display(df_ess_2020.head(4))
display(df_ess_2023.head(4))

,ID_Encuestado,Pais,Ronda,Genero,Anio_Nacimiento,Nivel_Educativo,Situacion_Economica,Voluntariado_12m,Donacion_Politica_12m,Capacidad_Rol_Politico,Confianza_ONU,Preocupacion_Clima,Responsabilidad_Clima,Inmigracion_Economia,Inmigracion_Cultura,Inmigracion_Pais_Mejor_Peor
0,27.0,AT,9,Hombre,1975.0,Secundaria alta (baja),Difícil,NaN,NaN,Poco capaz,5,NaN,NaN,5,6,0
1,137.0,AT,9,Hombre,1951.0,Secundaria alta (baja),Saliendo adelante,NaN,NaN,Poco capaz,2,NaN,NaN,5,4,5
2,194.0,AT,9,Mujer,1978.0,Secundaria baja,Cómodamente,NaN,NaN,Nada capaz,5,NaN,NaN,6,6,5
3,208.0,AT,9,Hombre,1955.0,Secundaria alta (baja),Saliendo adelante,NaN,NaN,Poco capaz,2,NaN,NaN,3,7,5


,ID_Encuestado,Pais,Ronda,Genero,Anio_Nacimiento,Nivel_Educativo,Situacion_Economica,Voluntariado_12m,Donacion_Politica_12m,Capacidad_Rol_Politico,Confianza_ONU,Preocupacion_Clima,Responsabilidad_Clima,Inmigracion_Economia,Inmigracion_Cultura,Inmigracion_Pais_Mejor_Peor
49519,10038.0,BE,10,Mujer,2006.0,Primaria o menos,Saliendo adelante,No,No,Nada capaz,NaN,Muy preocupado,5,NaN,8,9
49520,10053.0,BE,10,Mujer,1998.0,Máster/Doctorado,Cómodamente,No,No,Muy capaz,3,No muy preocupado,5,7,5,5
49521,10055.0,BE,10,Hombre,1964.0,Grado/BA,Saliendo adelante,No,No,Bastante capaz,8,Muy preocupado,8,6,5,5
49522,10062.0,BE,10,Hombre,1987.0,Secundaria alta (alta),Saliendo adelante,No,No,Nada capaz,10,No muy preocupado,NaN,8,8,8


,ID_Encuestado,Pais,Ronda,Genero,Anio_Nacimiento,Nivel_Educativo,Situacion_Economica,Voluntariado_12m,Donacion_Politica_12m,Capacidad_Rol_Politico,Confianza_ONU,Preocupacion_Clima,Responsabilidad_Clima,Inmigracion_Economia,Inmigracion_Cultura,Inmigracion_Pais_Mejor_Peor
109204,50014.0,AT,11,Hombre,1958.0,Secundaria alta (baja),Cómodamente,No,No,Completamente capaz,5,Muy preocupado,4,7,3,5
109205,50030.0,AT,11,Mujer,2002.0,Vocacional avanzado,Saliendo adelante,Sí,No,Poco capaz,5,Extremadamente preocupado,10,6,5,9
109206,50057.0,AT,11,Mujer,1970.0,Grado/BA,Cómodamente,Sí,Sí,Muy capaz,5,Extremadamente preocupado,8,9,9,8
109207,50106.0,AT,11,Mujer,1945.0,Vocacional avanzado,Saliendo adelante,No,No,Poco capaz,4,Muy preocupado,6,6,6,5


In [26]:
df_final.to_csv("clean-csv/ess_final.csv",index=False)
df_ess_2018.to_csv("clean-csv/ess_2018.csv",index=False)
df_ess_2020.to_csv("clean-csv/ess_2020.csv",index=False)
df_ess_2023.to_csv("clean-csv/ess_2023.csv",index=False)

